Projekt został zaczerpnięty ze strony kaggle.com . Generalnie również używa technik deep learning i do rozpoznawania obrazów używa tensorflow. Dokładność(accurancy) modelu jest nie mniejsza niż 90%. Komentarzy tym razem jest mniej bo by się powtarzały z wcześniejszych projektów.
https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import kagglehub
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, ReLU
from tensorflow.keras.models import Sequential

In [2]:
# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [3]:
train_path = f'{path}/chest_xray/train'
test_path = f'{path}/chest_xray/test'
val_path = f'{path}/chest_xray/val'
"""
train_ds = train_gen.flow_from_directory(f'{path}/chest_xray/train', target_size=(224,224),
                                         batch_size=32, class_mode='binary')
test_ds = train_gen.flow_from_directory(f'{path}/chest_xray/test', target_size=(224,224),
                                         batch_size=32, class_mode='binary')

val_ds = val_gen.flow_from_directory(f'{path}/chest_xray/val', target_size=(224,224),
                                     batch_size=32, class_mode='binary')
"""


"\ntrain_ds = train_gen.flow_from_directory(f'{path}/chest_xray/train', target_size=(224,224),\n                                         batch_size=32, class_mode='binary')\ntest_ds = train_gen.flow_from_directory(f'{path}/chest_xray/test', target_size=(224,224),\n                                         batch_size=32, class_mode='binary')\n\nval_ds = val_gen.flow_from_directory(f'{path}/chest_xray/val', target_size=(224,224),\n                                     batch_size=32, class_mode='binary')\n"

In [5]:
#print(dir(train_ds))

In [7]:
#train_ds

In [9]:
#print(train_ds.class_indices)

In [10]:
filepaths =[]
labels = []

folds = os.listdir(train_path)
for fold in folds:
    f_path = os.path.join(train_path , fold)
    filelists = os.listdir(f_path)
    for file in filelists:
        filepaths.append(os.path.join(f_path , file))
        labels.append(fold)

In [11]:
filepaths

['/kaggle/input/chest-xray-pneumonia/chest_xray/train/PNEUMONIA/person1180_virus_2010.jpeg',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/train/PNEUMONIA/person1230_virus_2081.jpeg',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/train/PNEUMONIA/person1513_virus_2632.jpeg',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/train/PNEUMONIA/person124_virus_238.jpeg',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/train/PNEUMONIA/person746_virus_1369.jpeg',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/train/PNEUMONIA/person588_bacteria_2422.jpeg',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/train/PNEUMONIA/person466_virus_960.jpeg',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/train/PNEUMONIA/person1590_bacteria_4175.jpeg',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/train/PNEUMONIA/person399_bacteria_1805.jpeg',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/train/PNEUMONIA/person59_bacteria_279.jpeg',
 '/kaggle/input/chest-xray-pneumonia/chest_xray/train/PNEUMONIA/pers

In [12]:
labels

['PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEUMONIA',
 'PNEU

In [13]:
filepaths_train = pd.Series(filepaths , name = 'filepaths')
labels_train = pd.Series(labels , name = 'label')

In [14]:
filepaths_train

,filepaths
0,/kaggle/input/chest-xray-pneumonia/chest_xray/...
1,/kaggle/input/chest-xray-pneumonia/chest_xray/...
2,/kaggle/input/chest-xray-pneumonia/chest_xray/...
3,/kaggle/input/chest-xray-pneumonia/chest_xray/...
4,/kaggle/input/chest-xray-pneumonia/chest_xray/...
...,...
5211,/kaggle/input/chest-xray-pneumonia/chest_xray/...
5212,/kaggle/input/chest-xray-pneumonia/chest_xray/...
5213,/kaggle/input/chest-xray-pneumonia/chest_xray/...
5214,/kaggle/input/chest-xray-pneumonia/chest_xray/...


In [15]:
labels_train

,label
0,PNEUMONIA
1,PNEUMONIA
2,PNEUMONIA
3,PNEUMONIA
4,PNEUMONIA
...,...
5211,NORMAL
5212,NORMAL
5213,NORMAL
5214,NORMAL


In [16]:
filepaths =[]
labels = []

folds = os.listdir(test_path)
for fold in folds:
    f_path = os.path.join(test_path , fold)
    filelists = os.listdir(f_path)
    for file in filelists:
        filepaths.append(os.path.join(f_path , file))
        labels.append(fold)

In [17]:
filepaths_test = pd.Series(filepaths , name = 'filepaths')
labels_test = pd.Series(labels , name = 'label')

In [18]:
filepaths_test

,filepaths
0,/kaggle/input/chest-xray-pneumonia/chest_xray/...
1,/kaggle/input/chest-xray-pneumonia/chest_xray/...
2,/kaggle/input/chest-xray-pneumonia/chest_xray/...
3,/kaggle/input/chest-xray-pneumonia/chest_xray/...
4,/kaggle/input/chest-xray-pneumonia/chest_xray/...
...,...
619,/kaggle/input/chest-xray-pneumonia/chest_xray/...
620,/kaggle/input/chest-xray-pneumonia/chest_xray/...
621,/kaggle/input/chest-xray-pneumonia/chest_xray/...
622,/kaggle/input/chest-xray-pneumonia/chest_xray/...


In [19]:
labels_test

,label
0,PNEUMONIA
1,PNEUMONIA
2,PNEUMONIA
3,PNEUMONIA
4,PNEUMONIA
...,...
619,NORMAL
620,NORMAL
621,NORMAL
622,NORMAL


In [20]:
#labels_train = np.array(labels_train).reshape(-1, 1)

In [21]:
"""
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
labels_train_encoded = encoder.fit_transform(labels_train)
labels_train_encoded
"""

"\nencoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)\nlabels_train_encoded = encoder.fit_transform(labels_train)\nlabels_train_encoded\n"

In [22]:
"""
le = LabelEncoder()
labels_train_encoded = le.fit_transform(labels_train)
"""

'\nle = LabelEncoder()\nlabels_train_encoded = le.fit_transform(labels_train)\n'

In [23]:
X_train, X_test, y_train, y_test = \
    train_test_split(filepaths_train, labels_train, test_size=0.2, random_state=42, stratify=labels_train)

In [24]:
train_df = pd.DataFrame({'filename': X_train, 'label': y_train})
val_df = pd.DataFrame({'filename': X_test, 'label': y_test})

In [25]:
train_df

,filename,label
3974,/kaggle/input/chest-xray-pneumonia/chest_xray/...,NORMAL
5159,/kaggle/input/chest-xray-pneumonia/chest_xray/...,NORMAL
3801,/kaggle/input/chest-xray-pneumonia/chest_xray/...,PNEUMONIA
24,/kaggle/input/chest-xray-pneumonia/chest_xray/...,PNEUMONIA
1308,/kaggle/input/chest-xray-pneumonia/chest_xray/...,PNEUMONIA
...,...,...
3140,/kaggle/input/chest-xray-pneumonia/chest_xray/...,PNEUMONIA
1063,/kaggle/input/chest-xray-pneumonia/chest_xray/...,PNEUMONIA
4580,/kaggle/input/chest-xray-pneumonia/chest_xray/...,NORMAL
2391,/kaggle/input/chest-xray-pneumonia/chest_xray/...,PNEUMONIA


In [26]:
# Dla treningu – z augmentacją
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=10,
                                   zoom_range=0.1,
                                   horizontal_flip=True)

# Dla walidacji – bez augmentacji
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    class_mode='binary',
    batch_size=32,
    shuffle=True
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    class_mode='binary',
    batch_size=32,
    shuffle=False
)

Found 4172 validated image filenames belonging to 2 classes.
Found 1044 validated image filenames belonging to 2 classes.


In [27]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)), # Changed input shape to 3 channels
    BatchNormalization(),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    Flatten(),

    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Output layer for binary classification
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
# Kompilacja
model.compile(optimizer='adam',
              loss='binary_crossentropy', # Loss function for binary classification
              metrics=['accuracy'])

In [29]:

history = model.fit(train_generator,
                    validation_data=val_generator,
                    epochs=10,
                    verbose=1)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 1957s 15s/step - accuracy: 0.8094 - loss: 1.4067 - val_accuracy: 0.8305 - val_loss: 0.5038
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 1968s 15s/step - accuracy: 0.8991 - loss: 0.2644 - val_accuracy: 0.8870 - val_loss: 0.2989
Epoch 3/10
  1/131 ━━━━━━━━━━━━━━━━━━━━ 32:15 15s/step - accuracy: 0.9688 - loss: 0.1095

KeyboardInterrupt: 